In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
import shap

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("../data/raw/Fraud_Data.csv")

# =========================
# 2. CLEAN DATA
# =========================
df = df.drop_duplicates()

# Convert datetime
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

# Feature engineering
df['time_diff'] = (
    df['purchase_time'] - df['signup_time']
).dt.total_seconds()

df = df.drop(['signup_time', 'purchase_time'], axis=1)

# =========================
# 3. REMOVE LEAKAGE COLUMNS
# =========================
df = df.drop(columns=[
    'user_id',
    'device_id',
    'ip_address'
], errors='ignore')

# =========================
# 4. SPLIT FEATURES/TARGET
# =========================
X = df.drop('class', axis=1)
y = df['class']

# =========================
# 5. ENCODE CATEGORICAL FEATURES
# =========================
cat_cols = X.select_dtypes(include='object').columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# =========================
# 6. CROSS VALIDATION SETUP
# =========================
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# =========================
# 7. MODEL
# =========================
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

# =========================
# 8. CROSS VALIDATION
# =========================
scores = cross_val_score(
    xgb_model,
    X,
    y,
    cv=skf,
    scoring='average_precision'
)

print("AUC-PR Scores:", scores)
print("Mean AUC-PR:", scores.mean())

# =========================
# 9. TRAIN FINAL MODEL
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

xgb_model.fit(X_train, y_train)

# =========================
# 10. EVALUATION
# =========================
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob))

# =========================
# 11. SHAP EXPLAINABILITY
# =========================
explainer = shap.Explainer(xgb_model)
shap_values = explainer(X_test)

# Global explanation
shap.summary_plot(shap_values, X_test)

# Feature importance
shap.summary_plot(shap_values, X_test, plot_type="bar")

ModuleNotFoundError: No module named 'xgboost'